# ptsrv.cloud

> Point-cloud loading, alignment, and cached E57 processing.

In [ ]:
#| default_exp cloud

In [ ]:
#| exporti
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import json
import os
import threading
from collections import OrderedDict
from dataclasses import dataclass, asdict, field
import numpy as np
import pye57

## Point cloud loading / caching for E57 files exported from Dot3D.

Dot3D writes one E57 "scan" per captured frame (hundreds of small scans, each
with its own pose).  pye57 applies the pose when reading, so we simply
concatenate all scans into one XYZ / RGB array.

The loaded cloud is cached next to the E57 as an .npz so subsequent API calls
are fast (numpy load of a 1M point cloud is ~50 ms vs ~0.6 s for E57 parse;
for 50M+ point clouds the difference is minutes vs seconds).


In [ ]:
#| export
CACHE_VERSION = 2

In [ ]:
#| export
@dataclass
class CloudInfo:
    name: str
    n_points: int
    xmin: float
    ymin: float
    zmin: float
    xmax: float
    ymax: float
    zmax: float
    floor_z: float          # detected floor level (world Z)
    ceiling_z: float        # detected ceiling level (world Z)
    rotation_deg: float     # rotation applied about Z to align walls with axes
    source: str
    extra: dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        return asdict(self)

In [ ]:
#| export
@dataclass
class Cloud:
    xyz: np.ndarray   # (N,3) float32, world coords (after optional alignment)
    rgb: np.ndarray   # (N,3) uint8
    info: CloudInfo

In [ ]:
#| export
def read_e57(path: str, max_points: int | None = None) -> tuple[np.ndarray, np.ndarray]:
    """Read every scan in an E57 and return (xyz float32 (N,3), rgb uint8 (N,3)).

    Poses are applied by pye57.  If the file has no colour, rgb is mid-grey.
    If max_points is set, the cloud is randomly subsampled to that size after
    load (keeps memory bounded on a small server).
    """

    e57 = pye57.E57(path)
    xs, cs = [], []
    for i in range(e57.scan_count):
        d = e57.read_scan(i, colors=True, intensity=False, ignore_missing_fields=True)
        n = len(d["cartesianX"])
        if n == 0:
            continue
        xyz = np.stack([d["cartesianX"], d["cartesianY"], d["cartesianZ"]], axis=1).astype(np.float32)
        if "colorRed" in d:
            rgb = np.stack([d["colorRed"], d["colorGreen"], d["colorBlue"]], axis=1).astype(np.uint8)
        else:
            rgb = np.full((n, 3), 128, np.uint8)
        xs.append(xyz)
        cs.append(rgb)
    if not xs:
        raise ValueError(f"No points found in {path}")
    xyz = np.concatenate(xs)
    rgb = np.concatenate(cs)
    # drop NaN / inf (some exporters emit invalid points)
    ok = np.isfinite(xyz).all(axis=1)
    if not ok.all():
        xyz, rgb = xyz[ok], rgb[ok]
    if max_points and len(xyz) > max_points:
        idx = np.random.default_rng(0).choice(len(xyz), max_points, replace=False)
        xyz, rgb = xyz[idx], rgb[idx]
    return xyz, rgb

In [ ]:
#| export
# ----------------------------------------------------------------------------
# Analysis helpers
# ----------------------------------------------------------------------------

In [ ]:
#| export
def detect_floor_ceiling(z: np.ndarray, bin_size: float = 0.02) -> tuple[float, float]:
    """Find floor and ceiling as the two strongest horizontal planes in the Z histogram.

    Floor = strongest peak in the lower half of the Z range, ceiling = strongest
    peak in the upper half.  Good enough for single storey interiors; for
    multi storey clouds call with a Z-filtered array or set levels manually.
    """
    lo, hi = np.percentile(z, [0.5, 99.5])
    bins = np.arange(lo, hi + bin_size, bin_size)
    h, edges = np.histogram(z, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    mid = 0.5 * (lo + hi)
    lower = centers < mid
    floor = float(centers[lower][np.argmax(h[lower])]) if lower.any() else float(lo)
    upper = ~lower
    ceiling = float(centers[upper][np.argmax(h[upper])]) if upper.any() else float(hi)
    return floor, ceiling

In [ ]:
#| export
def detect_wall_rotation(xyz: np.ndarray, floor_z: float, ceiling_z: float,
                         cell: float = 0.02) -> float:
    """Estimate the rotation (deg, about Z) that aligns the dominant walls with X/Y.

    Takes a horizontal band of the cloud, rasterises it to a density image,
    computes image gradients and finds the dominant gradient orientation
    modulo 90 degrees.  Returns the angle to rotate the cloud by (counter-
    clockwise positive).
    """
    band = (xyz[:, 2] > floor_z + 0.3) & (xyz[:, 2] < ceiling_z - 0.3)
    p = xyz[band][:, :2].astype(np.float64)
    if len(p) < 1000:
        return 0.0
    if len(p) > 300_000:
        p = p[np.random.default_rng(0).choice(len(p), 300_000, replace=False)]
    p -= p.mean(0)

    def score(deg: float) -> float:
        # Walls parallel to the axes concentrate points into few histogram bins;
        # sum of squared bin counts is maximal at the correct rotation.
        t = np.radians(deg)
        c, s = np.cos(t), np.sin(t)
        x = c * p[:, 0] - s * p[:, 1]
        y = s * p[:, 0] + c * p[:, 1]
        hx = np.bincount(np.floor((x - x.min()) / cell).astype(np.int64))
        hy = np.bincount(np.floor((y - y.min()) / cell).astype(np.int64))
        return float((hx.astype(np.float64) ** 2).sum() + (hy.astype(np.float64) ** 2).sum())

    coarse = np.arange(-45.0, 45.0, 1.0)
    best = coarse[int(np.argmax([score(a) for a in coarse]))]
    fine = np.arange(best - 1.0, best + 1.0001, 0.1)
    best = fine[int(np.argmax([score(a) for a in fine]))]
    finer = np.arange(best - 0.1, best + 0.1001, 0.02)
    best = finer[int(np.argmax([score(a) for a in finer]))]
    return float(round(best, 2))

In [ ]:
#| export
def rotate_z(xyz: np.ndarray, deg: float, about: tuple[float, float] = (0.0, 0.0)) -> np.ndarray:
    if abs(deg) < 1e-9:
        return xyz
    t = np.radians(deg)
    c, s = np.cos(t), np.sin(t)
    out = xyz.copy()
    x = xyz[:, 0] - about[0]
    y = xyz[:, 1] - about[1]
    out[:, 0] = c * x - s * y + about[0]
    out[:, 1] = s * x + c * y + about[1]
    return out

In [ ]:
#| export
def _cache_path(e57_path: str) -> str:
    return os.path.splitext(e57_path)[0] + ".scanplan.npz"

In [ ]:
#| export
def build_cloud(e57_path: str, align: str | float = "auto", max_points: int | None = None,
                name: str | None = None) -> Cloud:
    """Read an E57, detect floor/ceiling, optionally align walls, return a Cloud."""
    xyz, rgb = read_e57(e57_path, max_points=max_points)
    floor, ceiling = detect_floor_ceiling(xyz[:, 2])
    if align == "auto":
        rot = detect_wall_rotation(xyz, floor, ceiling)
    else:
        rot = float(align or 0.0)
    if rot:
        xyz = rotate_z(xyz, rot)
    mn, mx = xyz.min(0), xyz.max(0)
    info = CloudInfo(
        name=name or os.path.splitext(os.path.basename(e57_path))[0],
        n_points=int(len(xyz)),
        xmin=float(mn[0]), ymin=float(mn[1]), zmin=float(mn[2]),
        xmax=float(mx[0]), ymax=float(mx[1]), zmax=float(mx[2]),
        floor_z=floor, ceiling_z=ceiling, rotation_deg=rot,
        source=os.path.abspath(e57_path),
    )
    return Cloud(xyz=xyz, rgb=rgb, info=info)

In [ ]:
#| export
def load_cloud(e57_path: str, align: str | float = "auto", max_points: int | None = None,
               use_cache: bool = True) -> Cloud:
    """Load with on-disk cache.  Cache is invalidated when the E57 mtime changes."""
    cp = _cache_path(e57_path)
    src_mtime = os.path.getmtime(e57_path)
    if use_cache and os.path.exists(cp):
        try:
            with np.load(cp, allow_pickle=False) as z:
                meta = json.loads(str(z["meta"]))
                if meta.get("version") == CACHE_VERSION and abs(meta.get("src_mtime", -1) - src_mtime) < 1e-6 \
                        and meta.get("align") == str(align) and meta.get("max_points") == max_points:
                    info = CloudInfo(**meta["info"])
                    return Cloud(xyz=z["xyz"], rgb=z["rgb"], info=info)
        except Exception:
            pass  # corrupt cache -> rebuild
    cloud = build_cloud(e57_path, align=align, max_points=max_points)
    if use_cache:
        meta = {"version": CACHE_VERSION, "src_mtime": src_mtime, "align": str(align),
                "max_points": max_points, "info": cloud.info.to_dict()}
        tmp = cp[:-4] + ".tmp.npz"
        np.savez(tmp, xyz=cloud.xyz, rgb=cloud.rgb, meta=np.array(json.dumps(meta)))
        os.replace(tmp, cp)
    return cloud

In [ ]:
#| export
class CloudCache:
    """Small thread-safe in-memory LRU of loaded clouds for the API server."""

    def __init__(self, max_items: int = 2, **load_kwargs):
        self.max_items = max_items
        self.load_kwargs = load_kwargs
        self._items: OrderedDict[str, Cloud] = OrderedDict()
        self._lock = threading.Lock()

    def get(self, e57_path: str) -> Cloud:
        key = os.path.abspath(e57_path)
        with self._lock:
            if key in self._items:
                self._items.move_to_end(key)
                return self._items[key]
        cloud = load_cloud(e57_path, **self.load_kwargs)
        with self._lock:
            self._items[key] = cloud
            self._items.move_to_end(key)
            while len(self._items) > self.max_items:
                self._items.popitem(last=False)
        return cloud

    def evict(self, e57_path: str) -> None:
        with self._lock:
            self._items.pop(os.path.abspath(e57_path), None)